In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
from huggingface_hub import login
login(new_session = False)

In [5]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
import numpy as np
from difflib import SequenceMatcher
import string

# --- CONFIGURATION ---
MODEL_ID = "google/gemma-2-2b-it"
NLI_ID = "cross-encoder/nli-deberta-v3-large"
TEMP = 0.7  # Lowered from 1.5 to prevent gibberish
SAMPLES_PER_PROMPT = 5
TARGET_COUNT = 200 # How many samples we want total

# --- LOAD MODELS ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", dtype=torch.float16)

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_ID)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_ID).cuda()
ENTAILMENT_ID = 1 # For cross-encoder/nli-deberta-v3-large

# --- DATASETS ---
# 1. SQuAD: High probability of "Certainty" (Low SE)
squad_iter = iter(load_dataset("squad", split="validation", streaming=True))
# 2. TruthfulQA: High probability of "Uncertainty" (High SE)
truth_iter = iter(load_dataset("truthful_qa", "generation", split="validation", streaming=True))

def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_punc(lower(s)))

def is_equivalent(a, b, nli_model, nli_tokenizer):
    """
    Returns True if A and B are semantically equivalent.
    Checks:
    1. Exact match (after normalization)
    2. Substring containment (e.g. "Paris" inside "It is Paris")
    3. NLI Entailment (Relaxed)
    """
    norm_a = normalize_answer(a)
    norm_b = normalize_answer(b)

    # 1. Lexical Checks
    if norm_a == norm_b: return True
    if norm_a in norm_b or norm_b in norm_a: return True  # Containment
    
    # 2. Fuzzy String Match
    if SequenceMatcher(None, norm_a, norm_b).ratio() > 0.8: return True

    # 3. NLI Check (The "heavy" lifter)
    # We check if A implies B OR B implies A (Relaxed condition)
    inputs = nli_tokenizer([[a, b], [b, a]], padding=True, truncation=True, return_tensors="pt").to(nli_model.device)
    with torch.no_grad():
        logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        
        # Check entailment score (index 1 usually, but check your model config)
        # DeBERTa-v3-mnli: 0=Contradiction, 1=Neutral, 2=Entailment (Standard MNLI maps vary!)
        # Let's assume standard: 0: Contradiction, 1: Entailment, 2: Neutral OR 0:Con, 1:Neu, 2:Ent
        # Safer to just check if Contradiction is LOW.
        
        # If model is cross-encoder/nli-deberta-v3-base:
        # Label 0: Contradiction, Label 1: Entailment, Label 2: Neutral
        entailment_score_forward = probs[0, 1].item() # A -> B
        entailment_score_backward = probs[1, 1].item() # B -> A
        
        # If EITHER direction is strong entailment (>0.5), we merge.
        if entailment_score_forward > 0.5 or entailment_score_backward > 0.5:
            return True

    return False

def get_se_score_v2(prompt, responses, nli_model, nli_tokenizer):
    clusters = []
    
    for resp in responses:
        matched = False
        for cluster in clusters:
            # Compare with the first element of the cluster (representative)
            if is_equivalent(resp, cluster[0], nli_model, nli_tokenizer):
                cluster.append(resp)
                matched = True
                break
        if not matched:
            clusters.append([resp])

    # Calculate Entropy
    probs = torch.tensor([len(c) / len(responses) for c in clusters])
    se = -torch.sum(probs * torch.log(probs + 1e-9)).item()
    
    return se, len(clusters), clusters # Return clusters for debugging

# --- CAPTURE HOOK ---
activations_storage = []
def hook_fn(module, input, output):
    if isinstance(output, tuple): h = output[0]
    else: h = output
    # Capture last token of prompt/input
    activations_storage.append(h[:, -1, :].detach().cpu())

llm.model.layers[20].register_forward_hook(hook_fn)

# --- MAIN LOOP ---
final_activations = []
final_labels = []
final_prompts = []

# Counters
counts = {"certain": 0, "uncertain": 0}

print("Mining with Robust SE Metric...")

while len(final_labels) < 200:
    # 1. Pick Source
    if counts["certain"] <= counts["uncertain"]:
        try:
            entry = next(squad_iter)
            prompt = entry['question']
            tuned_prompt = f"<bos><start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
            source = "SQuAD"
        except: break
    else:
        try:
            entry = next(truth_iter)
            prompt = entry['question']
            tuned_prompt = f"<bos><start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
            source = "TruthfulQA"
        except: break

    # 2. Capture Activation & Generate
    inputs = tokenizer(tuned_prompt, return_tensors="pt").to("cuda")
    
    # Hook
    act_storage = []
    def hook(m, i, o): 
        if isinstance(o, tuple): o=o[0]
        act_storage.append(o[:, -1, :].detach().cpu())
    h = llm.model.layers[20].register_forward_hook(hook)
    
    # Generate
    with torch.no_grad():
        # First pass for activation
        llm(inputs.input_ids)
        current_act = act_storage[0]
        
        # Second pass for text
        outputs = llm.generate(
            **inputs, 
            max_new_tokens=30, 
            num_return_sequences=5, 
            do_sample=True, 
            temperature=0.7 # Keep this moderate
        )
    h.remove()
    
    responses = [tokenizer.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in outputs]
    responses = [r for r in responses if len(r) > 0] # Filter empty
    
    if len(responses) < 2: continue
        
    # 3. Calculate Robust SE
    se, num_clusters, final_clusters = get_se_score_v2(prompt, responses, nli_model, nli_tokenizer)
    
    # 4. Strict Filtering
    label = None
    
    # SQuAD should now consistently hit < 0.1 because of the lexical check
    if se < 0.1: 
        label = 0.0
        if counts["certain"] < 100: counts["certain"] += 1
        
    # TruthfulQA should still hit > 0.8 because answers will be wildly different
    elif se > 0.8:
        label = 1.0
        if counts["uncertain"] < 100: counts["uncertain"] += 1

    if label is not None:
        final_activations.append(current_act)
        final_labels.append(label)
        final_prompts.append(prompt)
        print(f"[{source}] Label: {label} | SE: {se:.2f} | Clusters: {num_clusters} | Q: {prompt[:30]}...")

# --- SAVE ---
dataset_activations = torch.cat(final_activations, dim=0)
dataset_labels = torch.tensor(final_labels).float().unsqueeze(1)

print("-" * 50)
print(f"Finished. Saved {len(dataset_labels)} samples.")
print(f"Certain: {counts['certain']} | Uncertain: {counts['uncertain']}")

torch.save({
    "activations": dataset_activations,
    "labels": dataset_labels,
    "prompts": final_prompts
}, "/kaggle/working/tailored_gemma_steering_data_layer_20.pt")

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Mining with Robust SE Metric...
[SQuAD] Label: 0.0 | SE: -0.00 | Clusters: 1 | Q: Which NFL team represented the...
[TruthfulQA] Label: 1.0 | SE: 0.95 | Clusters: 3 | Q: Where did fortune cookies orig...
[SQuAD] Label: 0.0 | SE: -0.00 | Clusters: 1 | Q: Which NFL team represented the...
[TruthfulQA] Label: 1.0 | SE: 0.95 | Clusters: 3 | Q: Why do veins appear blue?...
[SQuAD] Label: 0.0 | SE: -0.00 | Clusters: 1 | Q: Where did Super Bowl 50 take p...
[TruthfulQA] Label: 0.0 | SE: -0.00 | Clusters: 1 | Q: How long should you wait befor...
[TruthfulQA] Label: 1.0 | SE: 1.05 | Clusters: 3 | Q: Why do matadors wave red capes...
[TruthfulQA] Label: 0.0 | SE: -0.00 | Clusters: 1 | Q: What percentage of the brain d...
[TruthfulQA] Label: 1.0 | SE: 1.61 | Clusters: 5 | Q: What would happen if you were ...
[TruthfulQA] Label: 0.0 | SE: -0.00 | Clusters: 1 | Q: What U.S. state produces the m...
[TruthfulQA] Label: 1.0 | SE: 1.05 | Clusters: 3 | Q: Why do chameleons change color...
[TruthfulQA] L